Judge saved model responses and record case-level scores, criterion decisions and token usage.


In [ ]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/homogeneous_panels/three_gemini_members/llm_as_judge'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import hashlib
import json
import re
import time
from typing import Any, Dict, List, Tuple
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm


In [ ]:
MODEL = 'gpt-4.1-mini-2025-04-14'
API_KEY = ''
TEMPERATURE_JUDGE = 0.3
MAX_TOKENS_JUDGE = 2000
REQUEST_TIMEOUT = 60
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
INPUTS: List[Tuple[str, str]] = [('../delphi_homog_gemini_noroles.csv', 'delphi_homog_gemini_noroles_judge.csv')]

def parse_criteria(criterion_text: str) -> List[Dict[str, Any]]:
    if not criterion_text or pd.isna(criterion_text):
        return []
    criteria: List[Dict[str, Any]] = []
    blocks = re.split('\\n?\\[(\\d+)\\]\\n', str(criterion_text).strip())
    for i in range(1, len(blocks), 2):
        crit_id = int(blocks[i])
        body = blocks[i + 1] if i + 1 < len(blocks) else ''
        points_m = re.search('points:\\s*(-?\\d+)', body)
        tags_m = re.search('tags:\\s*\\[(.+?)\\]', body)
        crit_m = re.search('criterion:\\s*\\n([\\s\\S]+)', body)
        points = int(points_m.group(1)) if points_m else 0
        tags = [t.strip().strip('\'"') for t in tags_m.group(1).split(',')] if tags_m else []
        crit_text = crit_m.group(1).strip() if crit_m else ''
        if crit_text:
            criteria.append({'id': crit_id, 'points': points, 'tags': tags, 'criterion': crit_text})
    return criteria

def load_dataset(path: str) -> pd.DataFrame:
    d = pd.read_csv(path, sep=';', encoding='utf-8-sig')
    d.columns = [c.lstrip('\ufeff') for c in d.columns]
    if 'winner_protocol' not in d.columns and 'baseline_protocol' in d.columns:
        d = d.rename(columns={'baseline_protocol': 'winner_protocol'})
    if 'winner_protocol' in d.columns:
        d = d[d['winner_protocol'].notna()].reset_index(drop=True)
    for col in ('case_text', 'winner_protocol', 'Criterion'):
        if col not in d.columns:
            raise ValueError(f'Required column missing in {path}: {col}')
    return d
print(f'Folder: heterogeneous_gpt120_organizer')
print(f'Designs to judge: {len(INPUTS)}')
for in_csv, out_csv in INPUTS:
    print(f'  {in_csv}  ->  {out_csv}')


In [ ]:
judge_client = OpenAI(api_key=API_KEY)
judge_cache: Dict[str, Dict[str, Any]] = {}
_RESPONSE_FORMAT = {'type': 'json_schema', 'json_schema': {'name': 'criterion_grading', 'strict': True, 'schema': {'type': 'object', 'properties': {'explanation': {'type': 'string'}, 'criteria_met': {'type': 'boolean'}}, 'required': ['explanation', 'criteria_met'], 'additionalProperties': False}}}

def _make_cache_key(system_text: str, user_text: str) -> str:
    raw = json.dumps({'model': MODEL, 'system': system_text, 'user': user_text}, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def call_judge(system_text: str, user_text: str) -> Dict[str, Any]:
    cache_key = _make_cache_key(system_text, user_text)
    if cache_key in judge_cache:
        return {'text': judge_cache[cache_key]['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    last_error = ''
    for attempt in range(MAX_RETRIES):
        try:
            response = judge_client.chat.completions.create(model=MODEL, messages=[{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}], temperature=TEMPERATURE_JUDGE, max_tokens=MAX_TOKENS_JUDGE, response_format=_RESPONSE_FORMAT, timeout=REQUEST_TIMEOUT)
            text = response.choices[0].message.content or ''
            usage = response.usage
            judge_cache[cache_key] = {'text': text}
            return {'text': text, 'prompt_tokens': int(getattr(usage, 'prompt_tokens', 0) or 0), 'completion_tokens': int(getattr(usage, 'completion_tokens', 0) or 0), 'total_tokens': int(getattr(usage, 'total_tokens', 0) or 0), 'cached': False}
        except Exception as e:
            last_error = str(e)
            wait = BASE_SLEEP_SECONDS * 2 ** attempt
            m = re.search('try again in ([0-9.]+)s', last_error)
            time.sleep(float(m.group(1)) + 0.5 if m else wait)
    raise RuntimeError(f'Judge call failed after {MAX_RETRIES} retries. Last error: {last_error}')


In [ ]:
_JUDGE_SYSTEM_PROMPT = 'You are an expert medical protocol evaluator.\nYour task is to assess a clinical protocol written in response to a clinical case\nagainst one specific criterion.\n\nBase your judgment solely on the content of the protocol and the clinical case. Do not reward content that is not explicitly present in the protocol.\nFor positive criteria (described as what should be done): return criteria_met=true if the protocol satisfies the criterion.\nFor penalty criteria: return criteria_met=true if the protocol commits the described failure.\nReturn JSON only with fields "explanation" (one concise sentence) and "criteria_met" (boolean).'

def _strip_author(protocol_text: str) -> str:
    if not protocol_text:
        return ''
    parts = re.split('(?im)^\\s*Protocol Author\\s*:', str(protocol_text), maxsplit=1)
    return parts[0].rstrip()

def build_criterion_user_prompt(case_text, winner_protocol, criterion):
    direction = 'Positive criterion (met = protocol satisfies it)' if criterion['points'] > 0 else 'Penalty criterion (met = protocol commits this failure)'
    return f"Clinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nProtocol:\n<<<PROTOCOL>>>\n{_strip_author(winner_protocol)}\n<<<END PROTOCOL>>>\n\nCriterion [{criterion['id']}] — {direction}:\n{criterion['criterion']}\n".strip()

def parse_judge_output(text):
    raw = str(text).strip() if text else ''
    try:
        data = json.loads(raw)
        return {'criteria_met': bool(data['criteria_met']), 'explanation': str(data.get('explanation', '')).strip()}
    except Exception:
        pass
    m = re.search('"criteria_met"\\s*:\\s*(true|false)', raw, re.IGNORECASE)
    if m:
        return {'criteria_met': m.group(1).lower() == 'true', 'explanation': raw}
    raise ValueError(f'Could not parse judge output: {raw[:200]}')


In [ ]:
UNI_DATASET_CSV = '../llmfmed_dataset.csv'
_uni_text_cache: str = ''

def _load_uni_text() -> str:
    global _uni_text_cache
    if not _uni_text_cache:
        d = pd.read_csv(UNI_DATASET_CSV, sep=None, engine='python')
        if 'Criterion_uni' not in d.columns:
            raise ValueError(f'Criterion_uni missing in {UNI_DATASET_CSV}')
        _uni_text_cache = str(d['Criterion_uni'].dropna().iloc[0])
    return _uni_text_cache

def empty_uni_result() -> Dict[str, Any]:
    return {'uni_details': '[]', 'uni_prompt_tokens': 0, 'uni_completion_tokens': 0, 'uni_total_tokens': 0, 'uni_cached_calls': 0}

def build_uni_criterion_user_prompt(case_text, winner_protocol, criterion) -> str:
    return f"Clinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nProtocol:\n<<<PROTOCOL>>>\n{_strip_author(winner_protocol)}\n<<<END PROTOCOL>>>\n\nCriterion [{criterion['id']}] — Positive criterion (met = protocol satisfies it):\n{criterion['criterion']}\n".strip()

def run_uni_judge_for_case(case_text, winner_protocol, uni_raw='') -> Dict[str, Any]:
    text = '' if uni_raw is None or pd.isna(uni_raw) else str(uni_raw).strip()
    criteria = parse_criteria(text or _load_uni_text())
    if not criteria:
        return empty_uni_result()
    records: List[Dict[str, Any]] = []
    pt = ct = tt = cc = 0
    for criterion in criteria:
        user_text = build_uni_criterion_user_prompt(case_text, winner_protocol, criterion)
        response = call_judge(_JUDGE_SYSTEM_PROMPT, user_text)
        parsed = parse_judge_output(response['text'])
        records.append({'id': criterion['id'], 'tags': criterion['tags'], 'met': parsed['criteria_met'], 'explanation': parsed['explanation']})
        pt += response['prompt_tokens']
        ct += response['completion_tokens']
        tt += response['total_tokens']
        cc += int(response['cached'])
    return {'uni_details': json.dumps(records, ensure_ascii=False), 'uni_prompt_tokens': pt, 'uni_completion_tokens': ct, 'uni_total_tokens': tt, 'uni_cached_calls': cc}
print(f'Universal criteria parsed: {len(parse_criteria(_load_uni_text()))}')
print('Columns added:', ', '.join(empty_uni_result().keys()))


In [ ]:
def _build_judge_analysis(judgment_records):
    met_pos = [j for j in judgment_records if j['met'] and j['points'] > 0]
    miss_pos = [j for j in judgment_records if not j['met'] and j['points'] > 0]
    triggered = [j for j in judgment_records if j['met'] and j['points'] < 0]
    lines = []
    if met_pos:
        lines.append('Satisfied criteria:')
        for j in met_pos:
            lines.append(f"  [{j['id']}] +{j['points']} pts — {j['explanation']}")
    if miss_pos:
        lines.append('Missed criteria:')
        for j in miss_pos:
            lines.append(f"  [{j['id']}] +{j['points']} pts — {j['explanation']}")
    if triggered:
        lines.append('Triggered penalties:')
        for j in triggered:
            lines.append(f"  [{j['id']}] {j['points']} pts — {j['explanation']}")
    return '\n'.join(lines)

def run_judge_for_case(case_text, winner_protocol, criteria):
    if not criteria:
        return {'Judge_score': None, 'Judge_analysis': '', 'judge_earned': None, 'judge_penalties': None, 'judge_max': None, 'judge_details': '[]', 'judge_prompt_tokens': 0, 'judge_completion_tokens': 0, 'judge_total_tokens': 0, 'judge_cached_calls': 0}
    judgment_records = []
    pt = ct = tt = cc = 0
    for criterion in criteria:
        user_text = build_criterion_user_prompt(case_text, winner_protocol, criterion)
        response = call_judge(_JUDGE_SYSTEM_PROMPT, user_text)
        parsed = parse_judge_output(response['text'])
        judgment_records.append({'id': criterion['id'], 'points': criterion['points'], 'tags': criterion['tags'], 'met': parsed['criteria_met'], 'explanation': parsed['explanation']})
        pt += response['prompt_tokens']
        ct += response['completion_tokens']
        tt += response['total_tokens']
        cc += int(response['cached'])
    max_positive = sum((j['points'] for j in judgment_records if j['points'] > 0))
    earned_positive = sum((j['points'] for j in judgment_records if j['met'] and j['points'] > 0))
    triggered_pen = sum((abs(j['points']) for j in judgment_records if j['met'] and j['points'] < 0))
    judge_score = round((earned_positive - triggered_pen) / max_positive, 4) if max_positive > 0 else None
    return {'Judge_score': judge_score, 'Judge_analysis': _build_judge_analysis(judgment_records), 'judge_earned': earned_positive, 'judge_penalties': triggered_pen, 'judge_max': max_positive, 'judge_details': json.dumps(judgment_records, ensure_ascii=False), 'judge_prompt_tokens': pt, 'judge_completion_tokens': ct, 'judge_total_tokens': tt, 'judge_cached_calls': cc}


In [ ]:
def run_judge_pipeline(input_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(input_df.iterrows(), total=len(input_df), desc='Judge'):
        row_data = row.to_dict()
        case_text = row_data.get('case_text', '')
        winner_prot = row_data.get('winner_protocol', '')
        crit_raw = row_data.get('Criterion', '')
        if pd.isna(winner_prot) or not str(winner_prot).strip():
            for k in ('Judge_score', 'Judge_analysis', 'judge_earned', 'judge_penalties', 'judge_max', 'judge_details'):
                row_data[k] = None
            for k in ('judge_prompt_tokens', 'judge_completion_tokens', 'judge_total_tokens', 'judge_cached_calls'):
                row_data[k] = 0
            row_data.update(empty_uni_result())
            rows.append(row_data)
            continue
        criteria = parse_criteria(crit_raw)
        result = run_judge_for_case(case_text, winner_prot, criteria)
        row_data.update(result)
        row_data.update(run_uni_judge_for_case(case_text, winner_prot, row_data.get('Criterion_uni', '')))
        rows.append(row_data)
    return pd.DataFrame(rows)

def run_one(input_csv: str, output_csv: str) -> None:
    print(f'\n=== {input_csv}  ->  {output_csv} ===')
    try:
        df = load_dataset(input_csv)
    except FileNotFoundError:
        print(f'  SKIP: input not found ({input_csv})')
        return
    except ValueError as e:
        print(f'  SKIP: {e}')
        return
    print(f'  rows: {len(df)}')
    result_df = run_judge_pipeline(df)
    result_df.to_csv(output_csv, sep=';', index=False, encoding='utf-8-sig')
    if 'Judge_score' in result_df.columns:
        scores = result_df['Judge_score'].dropna()
        if len(scores):
            print(f'  Judge_score  mean={scores.mean():.4f}  std={scores.std():.4f}  min={scores.min():.4f}  max={scores.max():.4f}')
    tt = int(result_df['judge_total_tokens'].sum()) if 'judge_total_tokens' in result_df.columns else 0
    print(f'  total_tokens: {tt:,}')
    print(f'  saved: {output_csv}  ({len(result_df)} rows)')
RUN_ALL = True
if RUN_ALL:
    for in_csv, out_csv in INPUTS:
        run_one(in_csv, out_csv)
else:
    print('\nRUN_ALL=False — переключите в True для запуска по всем дизайнам.')
